# Dunnhumby M5_D = M2 + M3 + M4 — 마지막 주 10시드

**새로 학습하는 모델은 M5_D뿐입니다.** 기존 설계·계수 범위를 바꾸지 않고 seed42~51을 한 Colab에서 순차 실행합니다.

- M2: 확정 적응형 N/V·아이템 경제표현. M3: 중심화 N/V 그래프. M4: 고객 내 평균1 손실 가중치.
- 기존 M1·M2·M3·M4·M5_C의 같은 시드 결과50개를 재사용합니다. 참조모형 재학습·재평가, 기존 checkpoint를 이용한 초기화·동결은 없습니다.
- 새 학습10회, 각300epoch. 하나의 optimizer에서 모든 학습 파라미터를 공동 갱신합니다.
- 학습DAY1~704 / test DAY705~711. 검증·holdout·조기종료 없이 마지막 checkpoint에서만 평가합니다. test 결과로 구조·계수·epoch를 변경하지 않습니다.
- 학습 데이터는 앞 전체 기간, **CLV 산출은 기존 비교모형과 동일한 최근365일**입니다. 두 기간을 혼동하지 않습니다.
- ID64·2층·batch8192·lr0.0005·ID L2 0.001·shared L2 1e-6·uniform 음성1·MIN_ITEM_INTER=1. 학습 구매쌍은 평가 정답·후보에서 제외합니다.
- M5_A/B 실험과는 별도 출력입니다. 기존 A/B가 끝나기를 기다릴 필요가 없으며 그 실행을 변경하지 않습니다.

런타임을 **L4 GPU**로 선택하세요. 중단 후 같은 코드·경로로 재실행하면 완료모형은 건너뛰고 미완료모형은 저장 epoch부터 재개합니다. 로컬에서는 작은 입력 공동학습·재사용·집계를 검증했으며, 실제 Colab GPU/Drive 전체 실행은 아직 수행하지 않았습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
# Drive Errno107 발생 시 학습이 멈춘 상태에서 다음 줄 실행 후 준비 셀부터 재실행:
# drive.mount('/content/drive', force_remount=True)
assert torch.cuda.is_available(), 'L4 GPU를 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 {GPU}입니다. L4로 변경하세요.'
SOURCE_COMMIT = '890312be5040167915a17b74e31e3e72db22afe0'
REPO = Path('/content/clv-dh-m5-d-final10-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 참조 원본 검증·입력 준비 — 학습 없음

기존 Drive 결과 폴더에서 M2 및 M1/M3/M4/M5_C의10시드 원본을 찾습니다. 파일 SHA256·분할·시드·학습설정·전체142지표를 확인하며, 없거나 다르면 **학습 전에 멈춥니다. 대신 재학습하지 않습니다.**

필요하면 아래 두 경로에 Drive에 보관한 원본 ZIP을 지정할 수 있습니다. 재개할 때는 최초 경로를 유지하세요. 공통 데이터·M2 특징·M3 그래프·M4 가중치는 한 번만 준비합니다.

In [ ]:
import clv_dh_m5_d_adaptive_final10 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/.../clv_m2_adaptive_dunnhumby_final10_results.zip',
    # m4_reference='/content/drive/MyDrive/.../clv_m4_user_centered_final10_results.zip',
)
print(json.dumps(screen.asdict(cfg), ensure_ascii=False, indent=2))
prepared = screen.prepare(cfg)
print(json.dumps(screen.report(prepared)['reading'], ensure_ascii=False, indent=2))

## 2. M5_D 10개만 순차 학습

seed42 → 43 → … → 51. 각 모형300epoch 완료 후 test 평가·결과 저장을 마치고 다음 시드로 넘어갑니다. epoch마다 checkpoint를 저장합니다. 학습 중 아래 다른 셀을 실행하지 마세요.

순열 대조나 기존모형 학습은 포함하지 않습니다. 새 결과가 좋거나 나쁘더라도 일부 시드만 골라 최종 평균을 내지 않습니다.

In [ ]:
result = screen.run(prepared)
assert result['complete']
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 3. 전체 성능·시드별 차이 확인 — 추가 학습·평가 없음

완료된 저장 결과만 읽습니다. 중단 후 준비 셀을 실행하고 이 셀만 실행해도 됩니다.

- `absolute.csv`: 6모형×10시드×전체142지표 원값. 저/중/고CLV·노출 지표도 포함합니다.
- `summary.csv`: 전체142지표의 평균·**표본표준편차(ddof=1)**·최솟값·최댓값.
- `comparison.csv`: M5_D와 M1/M2/M3/M4/M5_C의 동일시드 대응 차이·변화율·증가/감소 시드 수.
- `paired_comparison.csv`: 모든 시드의 후보·참조 원값과 차이. M5_C 대비 비교는 M2 추가효과를 보는 직접 비교입니다.
- `diagnostics.csv`와 arm JSON: 작동 진단과1~300epoch 이력.

10시드 전체가 모였을 때만 평균±표준편차 최종표를 만듭니다. **표준편차는 신뢰구간이 아니며, 유의성·독립 확증·CLV 배정 귀속은 주장하지 않습니다.** 반복 관찰한 test의 한계를 유지합니다. 수치 증가가 모든 노출 지표에서 개선을 뜻하지는 않습니다.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
if result['complete']:
    print('10시드 평균 ± 표본표준편차 (신뢰구간 아님)')
    display(result['core_mean_std'])
    comparison = result['comparison']
    view = comparison[comparison['reference'].isin(['M1', 'M5_C']) & comparison['metric'].isin(screen.CORE)]
    display(view[['reference', 'metric', 'ratio_of_means_pct', 'paired_delta_mean', 'paired_delta_std',
                  'positive_delta_seeds', 'negative_delta_seeds', 'tie_seeds']])
    print('평균비율 변화(%) = 100 × (후보10시드평균 / 참조10시드평균 − 1). 유의성 주장 없음.')
else:
    print('미완료 — 현재 완료된 시드 원값만 표시합니다. 최종10시드 평균 아님.')
    display(result['absolute'][['model', 'seed', 'source'] + list(screen.CORE)])

## 4. 결과 ZIP 다운로드

전부 완료되면 `final10`, 일부 완료면 `partial`로 구분해 다운로드합니다. Drive에도 결과·checkpoint가 남습니다. 원본142지표·집계표·진단·protocol·참조 출처·새모형 학습이력을 포함하고, 큰 checkpoint는 ZIP에 넣지 않습니다. 학습 중에는 실행하지 마세요.

In [ ]:
from google.colab import files
result = screen.report(prepared)
suffix = 'final10' if result['complete'] else 'partial'
output = screen.export_zip(prepared, f'/content/dh_m5_d_adaptive_lastweek_{suffix}_results.zip',
                           allow_partial=not result['complete'])
files.download(output)